# Profiling the fractional-step solver on an A100

Before the paper compares the two formulations it has to compare them *fairly*.
The least-squares path went from 25.4 s/step to 1.62 s on an A100 through a
month of profiling; putting that beside a projection code nobody has profiled
would measure effort, not formulation.

**What a host-side run already says** (`colab/fs_profile.py`, numpy, 3 steps of
the consistent E path on the production mesh):

| phase | s/step | share | iterations/call |
|---|---|---|---|
| convective | 0.21 | 0.2 % | — |
| velocity Helmholtz | 0.67 | 0.8 % | 6 |
| **pressure (consistent E)** | **85.0** | **99.0 %** | **82** |

So the projection path is *one solve*, and everything else is noise. The
velocity Helmholtz is already excellent — 6 iterations under an FDM
preconditioner. The question for the A100 is whether that 99 % holds on a GPU,
and what can be done about it.

**Why E is hard, from its own source.** $E = G^{\mathsf T}M^{-1}G$ is a
mass-weighted Schur complement, not the assembled Laplacian $K$. A $K$-based
V-cycle preconditions it *badly* — 465 iterations against ~15 for $K$ on its own
system — which is why `lssem3d/epmg.py` builds the V-cycle on $E$ at every level.
That got it to 82. The K path, which solves $K$ instead, is 2–5× cheaper and is
the reason the two fractional-step variants differ in cost at all.


In [ ]:
#@title 1. GPU and CuPy
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
import subprocess
name = subprocess.run(['nvidia-smi','--query-gpu=name','--format=csv,noheader'],
                      capture_output=True, text=True).stdout.strip()
print('GPU:', name)
try:
    import cupy, numpy
    print('cupy', cupy.__version__, '| numpy', numpy.__version__)
except ImportError:
    print('installing cupy...')
    !pip install -q cupy-cuda12x
    import cupy; print('cupy', cupy.__version__)


In [ ]:
#@title 2. Code
import os
if os.path.isdir('/content/lssem/.git'):
    !cd /content/lssem && git fetch -q origin main && git reset -q --hard origin/main
else:
    !git clone -q --branch main https://github.com/chandc/Python_SEM.git /content/lssem
%cd /content/lssem
!git log --oneline -1
!pip install -q numba scipy matplotlib
print('\nvendored fractional-step tree:',
      len(__import__('glob').glob('fractional_step/**/*.py', recursive=True)), 'python files')


In [ ]:
#@title 3. Phase profile — the consistent (E) path
#@markdown 20 steps from the archived stationary field, restarting exactly as the
#@markdown production statistics run does.  The first step is excluded from the
#@markdown per-step figures (kernel compilation); `wall` includes everything.
STEPS = 20   #@param {type:"integer"}
TOLP  = '1e-4'  #@param {type:"string"}
!python colab/fs_profile.py --steps {STEPS} --backend cupy --consistent --tolp {TOLP}


In [ ]:
#@title 4. Phase profile — the weak-Laplacian (K) path, for contrast
#@markdown Same driver, same field, `K` instead of `E`: the pressure solve becomes
#@markdown a Helmholtz solve on the assembled Laplacian.  The cost difference
#@markdown between these two cells IS the price of controlling the weak
#@markdown divergence, measured rather than quoted.
!python colab/fs_profile.py --steps {STEPS} --backend cupy


In [ ]:
#@title 5. Is the E solve tolerance-bound or preconditioner-bound?
#@markdown Sweeping the pressure tolerance separates the two.  If iterations fall
#@markdown roughly as log(1/tol), the preconditioner is doing its job and the
#@markdown tolerance is the lever; if they barely move, the V-cycle is the lever
#@markdown and 1e-4 is already near its stagnation plateau -- which is what the
#@markdown E-run's stall at t~16 suggests.
for tol in ('1e-3', '1e-4', '1e-5'):
    print('='*72); print('tol_p =', tol, flush=True)
    !python colab/fs_profile.py --steps 8 --backend cupy --consistent --tolp {tol} 2>&1 | tail -9


## What to look for, and what each outcome implies

**If the pressure solve is still ~99 % on the GPU**, the projection path has
exactly one optimisation target and the rest of the code is irrelevant to its
cost. That is a cleaner situation than the least-squares path had.

**The tolerance sweep is the diagnostic that matters.** Iterations falling like
$\log(1/\mathrm{tol})$ means the V-cycle converges properly and $10^{-4}$ is a
choice; iterations flat or erratic means the V-cycle stagnates and the run is
living on its plateau — consistent with the E-run stopping near $t\approx16$
when the solves hit the iteration cap.

**Optimisation candidates, in the order I would try them.** Each is the direct
analogue of something already measured on the least-squares side:

1. **A vertex-patch smoother on $E$ instead of Chebyshev.** $E$'s coarse
   corrections miss for the same reason pointwise relaxation misses in the
   least-squares operator at large $c$ — the smoother cannot see the modes the
   coarse space cannot represent. The patch machinery is built, condensed and
   batched in the parent repository.
2. **The coarse level.** `epmg` assembles a dense $E$ per Fourier mode and
   inverts it by eigendecomposition. On the least-squares side, dropping the
   coarse degree to $p=1$ and holding the factor in fp32 gave 13× and 2× less
   memory for 5 % more iterations, and the coarse read was half the apply.
3. **Chebyshev degree.** Free to sweep, and the cheapest thing to try first.

Whatever comes out, the honest comparison is **hours per eddy turnover**, not
per step: the two schemes run at different time steps (3.5e−4 against 8e−4), so
a per-step number flatters whichever takes fewer of them.


In [ ]:
#@title 6. Cost per eddy turnover, both formulations, one machine
#@markdown The comparison metric.  Fill in the s/step measured above; the
#@markdown least-squares figure is the production run on this same GPU.
FS_SPS = 0.0   #@param {type:"number"}
FS_DT  = 3.5e-4  #@param {type:"number"}
LS_SPS, LS_DT = 1.62, 8e-4
if FS_SPS > 0:
    for lab, sps, dt in (('fractional step (E)', FS_SPS, FS_DT),
                         ('least squares', LS_SPS, LS_DT)):
        print(f'{lab:22s} {sps:6.3f} s/step x {1/dt:,.0f} steps = '
              f'{sps/dt/3600:5.2f} h per eddy turnover')
    print(f'\nratio: {(FS_SPS/FS_DT)/(LS_SPS/LS_DT):.2f}x '
          f'({"fractional step" if FS_SPS/FS_DT < LS_SPS/LS_DT else "least squares"} cheaper)')
    print('for reference, on the GB10 the same two were 6.07 and 8.82 h/turnover (1.45x)')
else:
    print('set FS_SPS from cell 3')
